# 03 · Detrending and finding periods

We can now get a clean-ish light curve. Two problems remain before a planet falls out: (1) the curve has slow **trends** that dwarf a transit, and (2) we need to find the period **automatically** instead of knowing it in advance.

**You'll learn:** what systematics/trends are and how **flattening** removes them · the difference between **Lomb–Scargle** and **BLS** periodograms · how to recover a period and confirm it by folding.

This notebook is the direct lead-in to the capstone, `kepler8b_transit_recovery.ipynb`.

In [ ]:
import numpy as np
import lightkurve as lk

search = lk.search_lightcurve('KIC 6922244', author='Kepler', cadence='long')
lc = search[1:5].download_all().stitch().remove_nans()
lc.plot();

## 1. Detrending (flattening)

Those slow rolls are **stellar variability** (starspots rotating in and out of view) and **instrument systematics** (temperature drifts, pointing changes) — all far larger than a ~1% transit. We remove them by fitting a smooth trend and dividing it out. `flatten()` does this with a Savitzky–Golay filter.

⚠️ **The one knob that bites people:** `window_length` must be **longer than a transit**. If the smoothing window is too short, the filter treats the transit itself as "trend" and irons it flat — deleting the very thing you're looking for. Here 901 cadences ≈ 19 days ≫ the few-hour transit, so we're safe.

In [ ]:
flat = lc.flatten(window_length=901)
flat.plot();

Now the curve sits flat at 1.0 and the transits are the dominant feature. (You may even spot the periodic dips by eye.)

## 2. Finding the period: periodograms

A **periodogram** tries many trial periods and scores how strongly the data repeats at each. Two you'll meet:

- **Lomb–Scargle** — looks for *sinusoidal* variations. Great for smooth things: pulsating or spotted stars, rotation. Wrong tool for transits, whose dips are sharp boxes, not sine waves.
- **Box Least Squares (BLS)** — looks specifically for a periodic *box-shaped dip*, spending most of the cycle flat with a brief drop. This is the transit-hunter's tool.

We ask BLS to score periods from 1–10 days. We do **not** tell it the answer — the peak *is* the discovery.

In [ ]:
period_grid = np.linspace(1, 10, 20000)
bls = flat.to_periodogram(method='bls', period=period_grid, frequency_factor=500)
bls.plot();

rec_period = bls.period_at_max_power.value
t0 = bls.transit_time_at_max_power.value
print(f'strongest period: {rec_period:.5f} days')

The single sharp spike is the orbital period the data prefers. (The smaller spikes at multiples/fractions are *aliases* — harmonics of the true period.)

## 3. Confirm by folding

A period from a periodogram is a *hypothesis*. We confirm it the way notebook 00 taught: fold on it and check that a clean, coherent transit appears.

In [ ]:
folded = flat.fold(period=rec_period, epoch_time=t0)
folded.scatter(s=1);

A clean V/U-shaped dip at phase 0 = the period is real. That's a detection.

## Recap
- Real curves need **detrending**; `flatten()` divides out slow trends — but keep `window_length` longer than a transit.
- **BLS** (box-shaped) is for transits; **Lomb–Scargle** (sinusoidal) is for smooth variability.
- A periodogram peak is a *hypothesis*; **folding** confirms it.

You now understand every step of the capstone. Open **`kepler8b_transit_recovery.ipynb`** to see this same pipeline run end-to-end *and validated against the published values* — including a cautionary bug (low-outlier clipping eating the transit) that shows why we check our answers.

## Learning resources
- 📗 [Lightkurve: removing systematics / flattening](https://docs.lightkurve.org/tutorials/2-creating-light-curves/2-3-removing-systematics.html)
- 📗 [Lightkurve: identifying transiting planets with BLS](https://docs.lightkurve.org/tutorials/3-science-examples/exoplanets-identifying-transiting-planet-signals.html)
- 📘 [Astropy: Box Least Squares](https://docs.astropy.org/en/stable/timeseries/bls.html)
- 🌍 [Lomb–Scargle periodogram](https://en.wikipedia.org/wiki/Lomb%E2%80%93Scargle_periodogram)
- 📄 [VanderPlas (2018), *Understanding the Lomb–Scargle Periodogram*](https://arxiv.org/abs/1703.09824) — excellent deep dive if you like primary sources
- 📄 [Kovács, Zucker & Mazeh (2002), the original BLS paper](https://arxiv.org/abs/astro-ph/0206099)

**Capstone:** `kepler8b_transit_recovery.ipynb`